<a href="https://colab.research.google.com/github/DhimanTarafdar/restoration-and-enhancement-ECG-Images/blob/main/ECG_image_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download sample ECG images (Dataset A + Dataset B) from GenECG dataset on Hugging Face.

- Dataset A -> clean ECG images (no imperfections) -> used as ground truth

- Dataset B -> same ECGs but with imperfections (looks like a photographed ECG) -> used as input

We only download a small random sample (not all 21,799 images) because that is enough

In [ ]:
!pip install -U huggingface_hub hf_xet

In [ ]:
import os
from huggingface_hub import login, hf_hub_download

# ------------------------------------------------------------------
# 1. Configuration & Setup
# ------------------------------------------------------------------
HF_TOKEN = "**************************"
login(token=HF_TOKEN)

REPO_ID = "edcci/GenECG"
NUM_SAMPLES = 100

DIR_A = "Dataset_A_ECGs_without_imperfections"
DIR_B = "Dataset_B_ECGs_with_imperfections"

os.makedirs(DIR_A, exist_ok=True)
os.makedirs(DIR_B, exist_ok=True)

# ------------------------------------------------------------------
# 2. Direct LFS Download Function
# ------------------------------------------------------------------
print(f"Downloading {NUM_SAMPLES} valid image pairs directly from Hugging Face LFS...\n")

success_count = 0

for i in range(1, NUM_SAMPLES + 1):
    # Generating relative file path inside subfolders (e.g., 00000/00001_hr_1R.png)
    file_id = f"{i:05d}"
    rel_path = f"00000/{file_id}_hr_1R.png"

    try:
        # Download Dataset A (Clean image)
        path_a = hf_hub_download(
            repo_id=REPO_ID,
            filename=f"{DIR_A}/{rel_path}",
            repo_type="dataset",
            token=HF_TOKEN,
            local_dir=".",
            local_dir_use_symlinks=False
        )

        # Download Dataset B (Imperfect image)
        path_b = hf_hub_download(
            repo_id=REPO_ID,
            filename=f"{DIR_B}/{rel_path}",
            repo_type="dataset",
            token=HF_TOKEN,
            local_dir=".",
            local_dir_use_symlinks=False
        )

        success_count += 1
        print(f"[{success_count}/{NUM_SAMPLES}] Successfully downloaded pair: {rel_path}")

    except Exception as e:
        print(f"Error downloading sample {file_id}: {e}")

# ------------------------------------------------------------------
# 3. Output Summary
# ------------------------------------------------------------------
print("\n--- Download Finished Successfully ---")
print(f"Total valid image pairs downloaded: {success_count}")
print(f"Clean Dataset Path: {DIR_A}/00000/")
print(f"Imperfect Dataset Path: {DIR_B}/00000/")

In [ ]:
import os
import shutil

# ------------------------------------------------------------------
# Save Processed Dataset Pairs as Zip Archive
# ------------------------------------------------------------------
ZIP_FILENAME = "GenECG_100_Pairs"

DIR_A = "Dataset_A_ECGs_without_imperfections"
DIR_B = "Dataset_B_ECGs_with_imperfections"

# Create a temporary container directory for clean archiving
temp_export_dir = "GenECG_Dataset_Sample"
os.makedirs(temp_export_dir, exist_ok=True)

# Copy folders to export directory
shutil.copytree(DIR_A, os.path.join(temp_export_dir, DIR_A), dirs_exist_ok=True)
shutil.copytree(DIR_B, os.path.join(temp_export_dir, DIR_B), dirs_exist_ok=True)

# Compress into zip format
shutil.make_archive(ZIP_FILENAME, 'zip', temp_export_dir)

# Clean up temporary folder
shutil.rmtree(temp_export_dir)

print(f"Dataset successfully packaged: {ZIP_FILENAME}.zip")

# ------------------------------------------------------------------
# Download to Local Machine (For Google Colab Environment)
# ------------------------------------------------------------------
try:
    from google.colab import files
    print("Initiating local browser download...")
    files.download(f"{ZIP_FILENAME}.zip")
except ImportError:
    print(f"Local machine users: Find '{ZIP_FILENAME}.zip' in your script root directory.")